In [1]:
import pandas as pd
import ast
import pandas as pd
from tqdm import tqdm
import numpy as np
from github_helper import from_github

In [2]:
df_votes = pd.DataFrame()
periods = [65, 66, 67, 68, 69, 70, 71]
for period in periods:
    df_period = pd.read_csv(from_github(f"/voting-data/votes_enriched_p{period}.csv"))
    print(f"Found {df_period["vote_id"].nunique()} votes by {df_period['aktørid'].nunique()} actors ({df_period["vote_id"].nunique() / df_period['aktørid'].nunique():.2f} avg.) on {df_period['afstemning_id'].nunique()} voting sessions in period {period}")
    df_votes = pd.concat([df_votes, df_period])

df_votes['all_topics'] = df_votes["all_topics"].apply(ast.literal_eval)
# df_votes.head()


Found 52626 votes by 189 actors (278.44 avg.) on 294 voting sessions in period 65
Found 227151 votes by 216 actors (1051.62 avg.) on 1269 voting sessions in period 66
Found 318096 votes by 241 actors (1319.90 avg.) on 1778 voting sessions in period 67
Found 309083 votes by 228 actors (1355.63 avg.) on 1727 voting sessions in period 68
Found 361192 votes by 237 actors (1524.02 avg.) on 2017 voting sessions in period 69
Found 302152 votes by 219 actors (1379.69 avg.) on 1688 voting sessions in period 70
Found 233758 votes by 235 actors (994.71 avg.) on 1306 voting sessions in period 71


In [3]:
# dd_mask = (df_votes['party'] == "Danmarksdemokraterne")
# mask_65 = (df_votes['Period'] == 65)

# df_votes[(dd_mask & mask_65)]#['politician'].unique()

In [4]:
#OKay so let's count the average number of votes per party per topic per year
# df_votes = pd.read_csv("-/voting-data")
df_votes.explode("all_topics").groupby(["Period", "party", "primary_topic"]).size().reset_index()
# votes_p.groupby()

,Period,party,primary_topic,0
0,65,Dansk Folkeparti,arbejdsmarked_velfærd,462
1,65,Dansk Folkeparti,bolig,132
2,65,Dansk Folkeparti,erhverv,1672
3,65,Dansk Folkeparti,finans_budget,11968
4,65,Dansk Folkeparti,forsvar_sikkerhed,264
...,...,...,...,...
1288,71,Venstre,social_familie,6026
1289,71,Venstre,sundhed,6808
1290,71,Venstre,transport_infrastruktur,2001
1291,71,Venstre,uddannelse,6900


In [5]:
def get_voting_percentages_for_party_on_voting_session(df):
    df_cast_votes = df[df["vote_typeid"] != 3]
    df_type_1 = df[df["vote_typeid"] == 1]
    df_type_2 = df[df["vote_typeid"] == 2]
    df_type_3 = df[df["vote_typeid"] == 3]
    df_type_4 = df[df["vote_typeid"] == 4]

    cols_to_group_by = ["afstemning_id", "party", "Period", "afstemning_vedtaget", "møde_year_month"]

    total_votes = (df.groupby(cols_to_group_by).size().rename("total_potential_votes"))
    cast_votes = (df_cast_votes.groupby(cols_to_group_by).size().rename("total_cast_votes"))

    votes_t1 = (df_type_1.groupby(cols_to_group_by).size().rename("votes_type_1"))
    votes_t2 = (df_type_2.groupby(cols_to_group_by).size().rename("votes_type_2"))
    votes_t3 = (df_type_3.groupby(cols_to_group_by).size().rename("votes_type_3"))
    votes_t4 = (df_type_4.groupby(cols_to_group_by).size().rename("votes_type_4"))

    voting_percentages = (
        total_votes.to_frame() #Create df for total_votes_shared
        .join([cast_votes, votes_t1, votes_t2, votes_t3, votes_t4], how = "left") #Left join as there are some people who shared votes but did not agree

        # .join(df_type_2, how = "left", on = ["voting_id", "party", "Period", "vedtaget", "Møde.dato"])
        .fillna(0)
        .reset_index() #Reset index releases the joined indexes so they become columns again
    )
    voting_percentages["agree_percentage_of_cast_votes"] = voting_percentages["votes_type_1"] / voting_percentages["total_cast_votes"]
    voting_percentages["disagree_percentage_of_cast_votes"] = voting_percentages["votes_type_2"] / voting_percentages["total_cast_votes"]
    voting_percentages["abstain_percentage_of_cast_votes"] = voting_percentages["votes_type_4"] / voting_percentages["total_cast_votes"]
    voting_percentages["absent_percentage_of_potential_votes"] = voting_percentages["votes_type_3"] / voting_percentages["total_potential_votes"]

    voting_percentages.fillna(0, inplace = True)
    voting_percentages.to_csv("./voting-data/voting_session_party_percentages.csv", index = False)
    return voting_percentages

voting_per = get_voting_percentages_for_party_on_voting_session(df_votes)
# voting_per.head()

In [6]:
def build_edges_for_period(filtered_df, period, period_col = "møde_year_month"):
    total_df_for_period = filtered_df[filtered_df[period_col] == period]
    df_period = total_df_for_period[['afstemning_id', 'vote_typeid', 'politician','party', period_col]] #, topic_col

    # #Now we find all pairs by joining the dataframe onto itself based on the 2 criteria.       
    pairs = df_period.merge(
        df_period
        , on = ["afstemning_id", period_col] # We do not group by the kind of vote they did, as we would rather do calculations on it seperately than create 3 df for each
        , suffixes = ("_source", "_target")
    )

    #Now we make sure we only have one observation per pair
    pairs = pairs[pairs['politician_source'] < pairs['politician_target']] #Only keep the ones where the politicians are different
    # 1) No duplicate pairs, so values are always different 
    # 2) No "reverse" pairs, as one has to be bigger than the other. If we had used != then there might have been (A,B) and (B,A)

    #Now we do something similar to ensure that we group the politician parties
    mask = pairs['party_source'] <= pairs['party_target']
    pairs['party_a'] = np.where(mask, pairs['party_source'], pairs['party_target'])
    pairs['party_b'] = np.where(mask, pairs['party_target'], pairs['party_source'])

    # #Now we have to count them.
    total_votes_shared_by_po = (
        pairs.groupby(["politician_source", "politician_target", "party_source", "party_target", period_col])
            .size() #Get the number of total votes shared 
            .rename("total_votes_shared")
    )

    total_votes_shared_by_pa = (
        pairs.groupby(['party_a', 'party_b', period_col])
        .size()
        .rename('total_votes_shared')
    )

    agreeing_pairs = pairs[pairs["vote_typeid_source"]==pairs["vote_typeid_target"]]
    agreed_votes_po = (
        agreeing_pairs.groupby(["politician_source", "politician_target"
                , "party_source", "party_target"
                , period_col]
            )
            .size()  
            .rename("total_votes_agreed")
    )

    agreed_votes_pa = (
        agreeing_pairs.groupby(["party_a", "party_b", period_col])
            .size()  
            .rename("total_votes_agreed")
    )

    #Calculate for the politician
    result_po = (
        total_votes_shared_by_po.to_frame() #Create df for total_votes_shared
        .join(agreed_votes_po, how = "left") #Left join as there are some people who shared votes but did not agree
        .fillna({'total_votes_agreed': 0})
        .reset_index() #Reset index releases the joined indexes so they become columns again
    )
    result_po['weight'] = result_po["total_votes_agreed"] / result_po["total_votes_shared"]

    #Calculate for the party
    result_pa = (
        total_votes_shared_by_pa.to_frame() #Create df for total_votes_shared
        .join(agreed_votes_pa, how = "left") #Left join as there are some people who shared votes but did not agree
        .fillna({'total_votes_agreed': 0})
        .reset_index()
    )
    result_pa['weight'] = result_pa["total_votes_agreed"] / result_pa["total_votes_shared"]

    return result_po, result_pa
# all_periods = df_votes["møde_year_month"].unique()
# period = all_periods[100]
# filtered_df = df_votes[df_votes['vote_typeid'] != 3]

# result_po, result_pa = build_edges_for_period(filtered_df, period, period_col="møde_year_month")


In [ ]:
def build_and_save_edges_for_multiple_periods(df, topic, topic_col = "all_topics", period_col = "møde_year_month"):
    filtered_df = df[df['vote_typeid'] != 3]
    if topic != "general":
        df_exp = filtered_df.explode(topic_col) #Explode if we are calculating it for a specific topic
        filtered_df = df_exp[df_exp[topic_col] == topic] #Limit the dataframe to only the chosen topic

    all_periods = filtered_df[period_col].unique()
    all_results_for_politicians = []
    all_results_for_parties = []
    print(f"Creating dataframe for {topic}, by {period_col}", end = " ")
    for period in all_periods:
        result_po, result_pa = build_edges_for_period(filtered_df, period, period_col)
        all_results_for_politicians.append(result_po)
        all_results_for_parties.append(result_pa)

    all_votes_po = pd.concat(all_results_for_politicians, ignore_index = True)
    all_votes_pa = pd.concat(all_results_for_parties, ignore_index = True)

    #Rename for using correct names
    all_votes_po.rename(
        columns = {
            'politician_source' : 'source'
            ,'politician_target' : 'target'
            ,'party_source' : 'source_party'
            ,'party_target' : 'target_party'
        }
        , inplace = True
    )

    all_votes_pa.rename(
        columns = {
            'party_a' : 'source'
            ,'party_b' : 'target'
        }
        , inplace= True
    )

    #Add the topics
    all_votes_po['topic'] = topic
    all_votes_pa['topic'] = topic

    all_votes_po.to_csv(f"./edges/politician/{period_col}/politician_edges_{topic}_by_{period_col}.csv", index = False)
    all_votes_pa.to_csv(f"./edges/party/{period_col}/party_edges_{topic}_by_{period_col}.csv", index = False)

    # return all_votes_po, all_votes_pa

# all_votes_po, all_votes_pa = build_edges_for_multiple_periods(df_votes, topic = "bolig", topic_col="all_topics", period_col = "møde_year_month")
# topic = "general"
# period_col = "møde_year_month"
# all_votes_po, all_votes_pa = build_edges_for_multiple_periods(df_votes, topic = topic, topic_col="all_topics", period_col = period_col)

In [ ]:
all_topics = df_votes.explode("all_topics")['all_topics'].dropna().unique() #Drop na, because it would otherwise return an "na" column, which we don't want
topics_for_loop = [topic for topic in all_topics]
topics_for_loop.append("general") 

for topic in tqdm(topics_for_loop):
    build_and_save_edges_for_multiple_periods(df_votes, topic = topic, topic_col="all_topics", period_col = 'Period')


########### Gorup it into just one big dataframe for the parties in periods
all_party_edges = pd.DataFrame()
for topic in topics_for_loop:
    party_df = pd.read_csv(f"./edges/party/Period/party_edges_{topic}_by_Period.csv")
    all_party_edges = pd.concat([all_party_edges, party_df])

all_party_edges.to_csv(f"./edges/party/all_party_edges_by_Period.csv", index = False)

  0%|          | 0/15 [00:00<?, ?it/s]

Creating dataframe for finans_budget, by Period 

  7%|▋         | 1/15 [00:48<11:23, 48.85s/it]

Creating dataframe for klima_miljø, by Period 

 13%|█▎        | 2/15 [01:12<07:21, 33.96s/it]

...

 87%|████████▋ | 13/15 [10:46<00:58, 29.41s/it]

Creating dataframe for bolig, by Period 

 93%|█████████▎| 14/15 [11:01<00:24, 24.90s/it]

Creating dataframe for general, by Period 

100%|██████████| 15/15 [13:00<00:00, 52.02s/it]
